**نسخه‌ی ویرایش‌شده:** توکن HF حذف شد؛ نمره‌های قبلی judge معتبرند (`RERUN_SFT=False`)؛ `judge_long` با بازیابی نمره از فیدبک‌ها جایگزین شد؛ مقایسه‌ی دو‌به‌دو روی سؤال‌های مشترک هر جفت. برچسب درستی: `em_strict`، `em_flex` و `em_best` (درست اگر هر یک از دو استخراج درست باشد؛ همان `exact_match` نهایی). `fmt_ok` = درصد پاسخ‌هایی که قالب «The answer is» را رعایت کرده‌اند.

In [3]:
!pip install -q transformers accelerate bitsandbytes

In [4]:
import os
from huggingface_hub import login

# Option A: Interactive login
login()

# Option B: Or directly via environment variable
# Option B: set HF_TOKEN via Kaggle Secrets / env var. Never paste the token in the notebook.

In [5]:
import torch, json, glob, os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "google/gemma-3-12b-it"   # verify the exact name on Hugging Face

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    device_map={"": 0},   # keep it on GPU 0
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
)
model.eval()

config.json:   0%|          | 0.00/916 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1065 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (embeddings): SiglipVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
        (position_embedding): Embedding(4096, 1152)
      )
      (encoder): SiglipEncoder(
        (layers): ModuleList(
          (0-26): 27 x SiglipEncoderLayer(
            (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (self_attn): SiglipAttention(
              (k_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (v_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (q_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
              (out_proj): Linear4bit(in_features=1152, out_features=1152, bias=True)
            )
            (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
            (mlp): SiglipMLP(
              (

In [6]:
TEMPLATE = """###Task Description:
An instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing an evaluation criteria are given.
1. Write a detailed feedback that assesses the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.
3. The output format should look as follows: "Feedback: (write a feedback for criteria) [RESULT] (an integer number between 1 and 5)"
4. Please do not generate any other opening, closing, and explanations.

###The instruction to evaluate:
{instruction}

###Response to evaluate:
{response}

###Reference Answer (Score 5):
{reference}

###Score Rubrics:
[{criteria}]
Score 1: {s1}
Score 2: {s2}
Score 3: {s3}
Score 4: {s4}
Score 5: {s5}

###Feedback:"""

RUBRIC = dict(
    criteria="Is the step-by-step reasoning correct, complete, and free of arithmetic errors, leading to the right final answer?",
    s1="The reasoning is mostly wrong and the final answer is wrong.",
    s2="The reasoning has major errors; the final answer is wrong.",
    s3="The reasoning is partly correct but has a notable error or gap; the final answer may be wrong.",
    s4="The reasoning is mostly correct with a minor flaw, and the final answer is correct.",
    s5="The reasoning is fully correct and clear, and the final answer is correct.",
)

SYSTEM_TEXT = "You are a fair evaluator language model."


def build_inputs(msg):
    """Gemma-3 has no system role, so the system text is merged into the user turn.
    apply_chat_template adds BOS and <start_of_turn> markers itself,
    so do NOT call tok(...) on the result again."""
    messages = [{"role": "user", "content": f"{SYSTEM_TEXT}\n\n{msg}"}]
    return tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)


@torch.no_grad()
def judge(item):
    msg = TEMPLATE.format(instruction=item["instruction"], response=item["response"],
                          reference=item["reference"], **RUBRIC)
    ids = build_inputs(msg)
    out = model.generate(**ids, do_sample=False, max_new_tokens=400)
    text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        score = int(text.split("[RESULT]")[1].strip()[0])
    except Exception:
        score = None
    return text, score

In [7]:
import numpy as np
import json, os

def load(path):
    """هر سؤال یک آیتم؛ هر دو فیلتر (strict و flexible) را نگه می‌داریم."""
    items = {}
    for line in open(path):
        r = json.loads(line)
        d = items.setdefault(r["doc_id"], {"instruction": r["doc"]["question"],
                                            "response": r["resps"][0][0].strip(),
                                            "reference": r["target"]})
        if r["filter"] == "strict-match":
            d["em_strict"] = r["exact_match"]
            # آیا پاسخ قالب «The answer is X» را رعایت کرده؟ (استخراج strict موفق بوده؟)
            fr = r.get("filtered_resps")
            while isinstance(fr, (list, tuple)) and fr:
                fr = fr[0]
            d["fmt_ok"] = int(str(fr).strip() != "[invalid]")
        elif r["filter"] == "flexible-extract":
            d["em_flex"] = r["exact_match"]
    for d in items.values():
        # قاعده‌ی ثابت برای همه‌ی مدل‌ها و همه‌ی سؤال‌ها: درست اگر هر یک از دو استخراج درست باشد.
        d["em_best"] = max(d.get("em_strict", 0), d.get("em_flex", 0))
        d["exact_match"] = d["em_best"]
    return items

model_files = {
    "base":        "/kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T18-13-42.977928.jsonl",
    "persian-grpo": "/kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T21-40-37.130015.jsonl",
    "english-grpo":      "/kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-08T08-20-12.599652.jsonl",
    "SFT":       "/kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T21-35-51.781761.jsonl", 
}

# چک کن همه‌ی فایل‌ها وجود دارند
for name, path in model_files.items():
    print(name, "OK" if os.path.exists(path) else "NOT FOUND", path)

runs = {name: load(path) for name, path in model_files.items()}
print({k: len(v) for k, v in runs.items()})   

ids = [set(v.keys()) for v in runs.values()]
assert ids[0] == ids[1] == ids[2] == ids[3], "سؤال‌ها یکسان نیستند"
print("OK: same questions in all runs")
# --- چک سلامت: دقت (exact_match) هر اجرا ---
# SFT_EXPECTED = دقت SFT در جدول اسلایدها برای gsm8k_cot (کل تست). روی ۱۰۰ سؤال چند درصد اختلاف طبیعی است.
SFT_EXPECTED = 0.56
for name, items in runs.items():
    st = np.mean([x.get("em_strict", np.nan) for x in items.values()])
    fl = np.mean([x.get("em_flex", np.nan) for x in items.values()])
    bs = np.mean([x["em_best"] for x in items.values()])
    fm = np.mean([x.get("fmt_ok", np.nan) for x in items.values()])
    print(f"{name}: strict = {st:.2f} | flexible = {fl:.2f} | best = {bs:.2f} | رعایت قالب = {fm:.0%}")
sft_em = np.mean([x["exact_match"] for x in runs["SFT"].values()])
if abs(sft_em - SFT_EXPECTED) > 0.15:
    print(f"\n!!! هشدار: دقت best SFT ({sft_em:.2f}) با اسلاید ({SFT_EXPECTED}) نمی‌خواند. مسیر فایل SFT در model_files را بررسی کنید.")


base OK /kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T18-13-42.977928.jsonl
persian-grpo OK /kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T21-40-37.130015.jsonl
english-grpo OK /kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-08T08-20-12.599652.jsonl
SFT OK /kaggle/input/datasets/arefehrajabian1379/qwen-logs/samples_gsm8k_cot_2026-10-25T21-35-51.781761.jsonl
{'base': 100, 'persian-grpo': 100, 'english-grpo': 100, 'SFT': 100}
OK: same questions in all runs
base: strict = 0.75 | flexible = 0.82 | best = 0.82 | رعایت قالب = 85%
persian-grpo: strict = 0.85 | flexible = 0.88 | best = 0.88 | رعایت قالب = 94%
english-grpo: strict = 0.86 | flexible = 0.86 | best = 0.86 | رعایت قالب = 98%
SFT: strict = 0.43 | flexible = 0.77 | best = 0.77 | رعایت قالب = 48%

!!! هشدار: دقت best SFT (0.77) با اسلاید (0.56) نمی‌خواند. مسیر فایل SFT در model_files را بررسی کنید.


In [8]:
for name, items in runs.items():
    text, score = judge(items[0])
    print(f"=== {name} | score={score}")
    print(text[:500], "\n")

=== base | score=5
The reasoning is fully correct and clear, and the final answer is correct. The response accurately calculates the number of eggs sold at the farmers' market by subtracting the number of eggs eaten and the number of eggs used for muffins from the total number of eggs laid. The final answer is also correct. [RESULT] 5 

=== persian-grpo | score=5
The reasoning is fully correct and clear, and the final answer is correct. The response accurately calculates the number of eggs used (3 + 4 = 7), subtracts that from the total number of eggs laid (16 - 7 = 9), and then correctly calculates the earnings from selling the eggs (9 * $2 = $18). [RESULT] 5 

=== english-grpo | score=5
The reasoning is fully correct and clear, and the final answer is correct. The response accurately calculates the number of eggs used (3 + 4 = 7), subtracts that from the total number of eggs laid (16 - 7 = 9), and then correctly calculates the earnings from selling the eggs (9 * $2 = $18). [RESULT] 5

In [9]:
OUT = "/kaggle/working/judge_results3.json"
results = json.load(open(OUT)) if os.path.exists(OUT) else {}

# judge فقط به متن پاسخ وابسته است، نه به برچسب درستی؛ اگر فایل SFT را عوض نکرده‌اید، نمره‌های قبلی معتبرند.
# فقط اگر مسیر SFT را به فایل دیگری تغییر داده‌اید True بگذارید.
RERUN_SFT = False
if RERUN_SFT:
    results = {k: v for k, v in results.items() if not k.startswith("SFT|")}

for run, items in runs.items():
    for i, item in items.items():
        key = f"{run}|{i}"
        if key in results: continue              # بعد از قطع شدن ادامه می‌دهد
        text, score = judge(item)
        results[key] = {"score": score, "feedback": text, "exact_match": item["exact_match"]}
        if len(results) % 10 == 0:
            json.dump(results, open(OUT, "w"))
            print(len(results), "/ 400", end="\r")
json.dump(results, open(OUT, "w"))
print("done:", len(results))                      # باید 400 باشد

done: 400


In [10]:
# بازیابی نمره از فیدبک‌های ذخیره‌شده (بدون GPU و بدون اجرای دوباره‌ی judge)
# قبلاً judge_long با ۷۰۰ توکن هیچ نمره‌ای را بازیابی نکرد؛ مشکل قالب خروجی است نه طول.
import re

def parse_score(text):
    m = re.findall(r"\[RESULT\]\s*\(?([1-5])", text)
    if m:
        return int(m[-1]), "result_tag"
    m = re.findall(r"score\s*(?:is|of|:)\s*\(?([1-5])\b", text, flags=re.I)
    return (int(m[-1]), "fallback") if m else (None, "failed")

rec = 0
for v in results.values():
    if v["score"] is None:
        s, how = parse_score(v["feedback"])
        if s is not None:
            v["score"], v["parsed_by"] = s, how
            rec += 1
json.dump(results, open(OUT, "w"))
print("بازیابی‌شده:", rec, "| باقی‌مانده:", sum(v["score"] is None for v in results.values()))


بازیابی‌شده: 0 | باقی‌مانده: 106


In [11]:
import pandas as pd, numpy as np
from itertools import combinations
from scipy import stats

df = pd.DataFrame([
    {"run": k.split("|")[0], "doc": int(k.split("|")[1]), **v}
    for k, v in results.items()
])

df["exact_match"] = [runs[r][d]["exact_match"] for r, d in zip(df["run"], df["doc"])]
df["em_strict"] = [runs[r][d].get("em_strict") for r, d in zip(df["run"], df["doc"])]
df["em_flex"] = [runs[r][d].get("em_flex") for r, d in zip(df["run"], df["doc"])]
df["em_best"] = [runs[r][d]["em_best"] for r, d in zip(df["run"], df["doc"])]
df["fmt_ok"] = [runs[r][d].get("fmt_ok") for r, d in zip(df["run"], df["doc"])]

print("parse failures per run:")
print(df[df["score"].isna()].groupby("run").size(), "\n")

summary = df.groupby("run").agg(
    judge_mean=("score", "mean"),
    judge_std=("score", "std"),
    em_strict=("em_strict", "mean"),
    em_flex=("em_flex", "mean"),
    em_best=("em_best", "mean"),
    fmt_ok=("fmt_ok", "mean"),
    n_scored=("score", "count"),
)
print(summary, "\n")

print("میانگین نمره‌ی judge بر اساس درست/غلط بودن (exact_match):")
print(df.groupby("exact_match")["score"].mean())
print("\nنرخ شکست پارس برای هر اجرا:")
print(df["score"].isna().groupby(df["run"]).mean().round(3))
if "parsed_by" in df.columns:
    print("\nروش پارس (فقط ردیف‌های بازیابی‌شده):")
    print(df["parsed_by"].value_counts())


parse failures per run:
run
SFT              9
base            38
english-grpo    30
persian-grpo    29
dtype: int64 

              judge_mean  judge_std  em_strict  em_flex  em_best  fmt_ok  \
run                                                                        
SFT             4.736264   0.663914       0.43     0.77     0.77    0.48   
base            4.677419   0.695988       0.75     0.82     0.82    0.85   
english-grpo    4.685714   0.713084       0.86     0.86     0.86    0.98   
persian-grpo    4.633803   0.701823       0.85     0.88     0.88    0.94   

              n_scored  
run                     
SFT                 91  
base                62  
english-grpo        70  
persian-grpo        71   

میانگین نمره‌ی judge بر اساس درست/غلط بودن (exact_match):
exact_match
0.0    3.842105
1.0    4.890295
Name: score, dtype: float64

نرخ شکست پارس برای هر اجرا:
run
SFT             0.09
base            0.38
english-grpo    0.30
persian-grpo    0.29
Name: score, dtype: float

In [12]:
p_all = df.pivot(index="doc", columns="run", values="score")
print("تعداد نمره‌ی موجود برای هر اجرا:")
print(p_all.notna().sum(), "\n")

# آزمون کلی (Friedman) فقط روی سؤال‌هایی که هر 4 اجرا نمره دارند
p = p_all.dropna()
print("سؤال‌هایی که برای هر 4 اجرا نمره دارند:", len(p))
print(stats.friedmanchisquare(*[p[c] for c in p.columns]), "\n")

# مقایسه‌ی دو به دو: فقط سؤال‌های مشترک همان دو اجرا (نه هر 4 تا)
rng = np.random.default_rng(0)
for a, b in combinations(p_all.columns, 2):
    d = (p_all[a] - p_all[b]).dropna().values
    boots = [rng.choice(d, len(d)).mean() for _ in range(5000)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"{a} vs {b}: n={len(d)} wins {(d>0).sum()}/{(d<0).sum()} ties {(d==0).sum()}"
          f" | mean diff {d.mean():+.2f}  CI [{lo:+.2f}, {hi:+.2f}]")


تعداد نمره‌ی موجود برای هر اجرا:
run
SFT             91
base            62
english-grpo    70
persian-grpo    71
dtype: int64 

سؤال‌هایی که برای هر 4 اجرا نمره دارند: 46
FriedmanchisquareResult(statistic=np.float64(9.240000000000087), pvalue=np.float64(0.026264417987089624)) 

SFT vs base: n=57 wins 9/1 ties 47 | mean diff +0.18  CI [+0.05, +0.32]
SFT vs english-grpo: n=65 wins 5/2 ties 58 | mean diff +0.06  CI [-0.03, +0.15]
SFT vs persian-grpo: n=67 wins 7/2 ties 58 | mean diff +0.10  CI [-0.03, +0.24]
base vs english-grpo: n=50 wins 3/6 ties 41 | mean diff -0.06  CI [-0.18, +0.06]
base vs persian-grpo: n=50 wins 3/7 ties 40 | mean diff -0.08  CI [-0.20, +0.04]
english-grpo vs persian-grpo: n=63 wins 0/1 ties 62 | mean diff -0.02  CI [-0.05, +0.00]


In [13]:
em_all = {lab: df.pivot(index="doc", columns="run", values=lab) for lab in ["em_strict", "em_flex", "em_best"]}

for lab, em in em_all.items():
    print(f"\n=== McNemar با برچسب {lab}")
    for a, b in combinations(em.columns, 2):
        only_a = int(((em[a] == 1) & (em[b] == 0)).sum())
        only_b = int(((em[a] == 0) & (em[b] == 1)).sum())
        m = only_a + only_b
        pv = stats.binomtest(min(only_a, only_b), m, 0.5).pvalue if m else 1.0
        print(f"{a} vs {b}: only {a}={only_a}, only {b}={only_b}, McNemar p={pv:.3f}")



=== McNemar با برچسب em_strict
SFT vs base: only SFT=4, only base=36, McNemar p=0.000
SFT vs english-grpo: only SFT=1, only english-grpo=44, McNemar p=0.000
SFT vs persian-grpo: only SFT=2, only persian-grpo=44, McNemar p=0.000
base vs english-grpo: only base=2, only english-grpo=13, McNemar p=0.007
base vs persian-grpo: only base=3, only persian-grpo=13, McNemar p=0.021
english-grpo vs persian-grpo: only english-grpo=4, only persian-grpo=3, McNemar p=1.000

=== McNemar با برچسب em_flex
SFT vs base: only SFT=7, only base=12, McNemar p=0.359
SFT vs english-grpo: only SFT=5, only english-grpo=14, McNemar p=0.064
SFT vs persian-grpo: only SFT=6, only persian-grpo=17, McNemar p=0.035
base vs english-grpo: only base=2, only english-grpo=6, McNemar p=0.289
base vs persian-grpo: only base=2, only persian-grpo=8, McNemar p=0.109
english-grpo vs persian-grpo: only english-grpo=2, only persian-grpo=4, McNemar p=0.688

=== McNemar با برچسب em_best
SFT vs base: only SFT=7, only base=12, McNemar p

In [14]:
from sklearn.metrics import roc_auc_score
d = df.dropna(subset=["score"])

for lab in ["em_strict", "em_flex", "em_best"]:
    print(f"\n=== برچسب درستی: {lab}")
    print("AUC کلی (judge vs برچسب):", round(roc_auc_score(d[lab], d["score"]), 3))
    for name, g in d.groupby("run"):
        if g[lab].nunique() == 2:
            print(f"  {name}: {roc_auc_score(g[lab], g['score']):.3f}")

wrong_high = d[(d["exact_match"] == 0) & (d["score"] >= 4)]
print("\nغلط (بر اساس exact_match نهایی) با نمره >= 4:", len(wrong_high))
for _, r in wrong_high.head(3).iterrows():
    print(r["run"], r["doc"], r["score"]); print(r["feedback"][:400]); print("-----")



=== برچسب درستی: em_strict
AUC کلی (judge vs برچسب): 0.663
  SFT: 0.544
  base: 0.722
  english-grpo: 0.891
  persian-grpo: 0.863

=== برچسب درستی: em_flex
AUC کلی (judge vs برچسب): 0.787
  SFT: 0.669
  base: 0.775
  english-grpo: 0.891
  persian-grpo: 0.934

=== برچسب درستی: em_best
AUC کلی (judge vs برچسب): 0.787
  SFT: 0.669
  base: 0.775
  english-grpo: 0.891
  persian-grpo: 0.934

غلط (بر اساس exact_match نهایی) با نمره >= 4: 29
base 12 4.0
The reasoning is mostly correct, but there's a minor flaw in the calculation. The response correctly calculates the annual earnings ($10.5 - $3 = $7.5) and the initial cost ($90). However, the final answer is calculated incorrectly. To determine the number of years to recoup the initial investment, we need to divide the initial cost by the net annual earnings: $90 / $7.5 = 12. However, since the q
-----
base 36 5.0
The reasoning is fully correct and clear, and the final answer is correct. The response correctly calculates the total number of y

In [15]:
import pandas as pd
rows = []
for k, v in results.items():
    run, i = k.split("|"); i = int(i)
    it = runs[run][i]
    rows.append(dict(run=run, doc=i, judge=v["score"], em=it["exact_match"],
                     question=it["instruction"], response=it["response"],
                     reference=it["reference"], feedback=v["feedback"]))
D = pd.DataFrame(rows)

print("--- درست و نمره ۵"); print(D[(D.em==1)&(D.judge==5)].groupby("run").head(2)[["run","doc"]])
print("--- غلط و نمره ≤۲");  print(D[(D.em==0)&(D.judge<=2)].groupby("run").head(2)[["run","doc"]])
print("--- غلط و نمره ≥۴");  print(D[(D.em==0)&(D.judge>=4)].groupby("run").head(2)[["run","doc"]])

parts = [g.sample(min(20, len(g)), random_state=0) for _, g in D[D.em==0].groupby("run")]
lab = pd.concat(parts); lab["category"] = ""
lab.to_csv("/kaggle/working/error_labeling_qwen.csv", index=False, encoding="utf-8-sig")
print(lab.groupby("run").size())

--- درست و نمره ۵
              run  doc
0            base    0
3            base    6
100  persian-grpo    0
102  persian-grpo    4
200  english-grpo    0
202  english-grpo    4
302           SFT    4
303           SFT    6
--- غلط و نمره ≤۲
              run  doc
268  english-grpo   37
368           SFT   37
--- غلط و نمره ≥۴
              run  doc
6            base   12
18           base   36
106  persian-grpo   12
139  persian-grpo   78
206  english-grpo   12
218  english-grpo   36
300           SFT    0
313           SFT   26
run
SFT             20
base            18
english-grpo    14
persian-grpo    12
dtype: int64
